In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import models
import numpy as np
import random
import os

# ---- Set Seed for Reproducibility ----
SEED = 42

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

np.random.seed(SEED)
random.seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed set.")

# ---- Device Setup ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Seed set.
Using device: cuda


In [2]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler

DATA_DIR = "data/mri-data"
IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

full_dataset = torchvision.datasets.ImageFolder(
    f"{DATA_DIR}/full_dataset",
    transform=train_transform
)

# Get labels from dataset
labels = [label for _, label in full_dataset]

# Count class frequencies
counts = Counter(labels)
print("Class distribution:", counts)

# Create weights for each sample
weights = [1.0 / counts[label] for label in labels]

# Weighted sampler
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# DataLoader with sampler
train_loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler
)

Class distribution: Counter({2: 3200, 3: 2235, 0: 896, 1: 64})


In [3]:
model = models.resnet18(weights="IMAGENET1K_V1")

# Replace final layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 4)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last residual block
for param in model.layer4.parameters():
    param.requires_grad = True

# Unfreeze classifier layer
for param in model.fc.parameters():
    param.requires_grad = True

# Move model to device
model = model.to(device)

print("Layer4 + FC unfrozen. Model ready.")

Layer4 + FC unfrozen. Model ready.


In [4]:
class_weights = torch.tensor([1.5, 3.0, 1.0, 1.2], dtype=torch.float).to(device)
print("Using manual class weights:", class_weights)
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    list(model.layer4.parameters()) + list(model.fc.parameters()),
    lr=0.0001
)

print("Loss and optimizer ready.")

Using manual class weights: tensor([1.5000, 3.0000, 1.0000, 1.2000], device='cuda:0')
Loss and optimizer ready.


In [5]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {running_loss/len(train_loader):.4f}")
    print(f"Train Acc: {train_acc:.2f}%")

# ---- Save Final Model ----
torch.save(model.state_dict(), "models/final_model.pth")
print("\nFinal model saved at models/final_model.pth")


Epoch 1
Loss: 0.6855
Train Acc: 69.57%

Epoch 2
Loss: 0.4685
Train Acc: 80.53%

Epoch 3
Loss: 0.3645
Train Acc: 85.39%

Epoch 4
Loss: 0.3031
Train Acc: 88.01%

Epoch 5
Loss: 0.2503
Train Acc: 89.95%

Epoch 6
Loss: 0.2126
Train Acc: 91.95%

Epoch 7
Loss: 0.1864
Train Acc: 92.99%

Epoch 8
Loss: 0.1619
Train Acc: 93.68%

Epoch 9
Loss: 0.1434
Train Acc: 94.61%

Epoch 10
Loss: 0.1460
Train Acc: 94.54%

Final model saved at models/final_model.pth


In [12]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

                  precision    recall  f1-score   support

    MildDemented       0.77      0.72      0.74       180
ModerateDemented       1.00      0.77      0.87        13
     NonDemented       0.88      0.77      0.82       640
VeryMildDemented       0.69      0.84      0.76       443

        accuracy                           0.79      1276
       macro avg       0.84      0.77      0.80      1276
    weighted avg       0.80      0.79      0.79      1276

